# Лабораторная работа 9

## Асинхронность и Telegram-боты

In [1]:
# flake8: noqa
import asyncio
import json
import ssl
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import certifi

SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())


def fetch_json_url(url, params=None):
    if params:
        url = f"{url}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": "Lab9/1.0"})
    with urlopen(request, timeout=20, context=SSL_CONTEXT) as response:
        return json.loads(response.read().decode("utf-8"))


def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    return loop.create_task(coro)


## Асинхронность

### №1
Имеется синхронная версия программы для подсчета факториала чисел. Используя
модуль `asyncio`, сделаем программу асинхронной.

In [2]:
# flake8: noqa
def factorial_sync(name, number):
    f = 1
    for i in range(2, number + 1):
        print(f"Task {name}: Compute factorial({i})...")
        f *= i
    print(f"Task {name}: factorial({number}) = {f}")


async def factorial(name, number):
    f = 1
    for i in range(2, number + 1):
        print(f"Task {name}: Compute factorial({i})...")
        f *= i
        await asyncio.sleep(0)
    print(f"Task {name}: factorial({number}) = {f}")
    return f


async def run_factorials():
    return await asyncio.gather(
        factorial("A", 15),
        factorial("B", 7),
        factorial("C", 4),
    )


run_async(run_factorials())


<Task pending name='Task-31' coro=<run_factorials() running at /var/folders/kp/d5m_y7fs5f9dzqchhd9s80hw0000gn/T/ipykernel_54422/631097123.py:20>>

Task A: Compute factorial(2)...
Task B: Compute factorial(2)...
Task C: Compute factorial(2)...
Task A: Compute factorial(3)...
Task B: Compute factorial(3)...
Task C: Compute factorial(3)...
Task A: Compute factorial(4)...
Task B: Compute factorial(4)...
Task C: Compute factorial(4)...
Task A: Compute factorial(5)...
Task B: Compute factorial(5)...
Task C: factorial(4) = 24
Task A: Compute factorial(6)...
Task B: Compute factorial(6)...
Task A: Compute factorial(7)...
Task B: Compute factorial(7)...
Task A: Compute factorial(8)...
Task B: factorial(7) = 5040
Task A: Compute factorial(9)...
Task A: Compute factorial(10)...
Task A: Compute factorial(11)...
Task A: Compute factorial(12)...
Task A: Compute factorial(13)...
Task A: Compute factorial(14)...
Task A: Compute factorial(15)...
Task A: factorial(15) = 1307674368000


Задачи выполняются вперемешку, потому что после каждой итерации уступают
управление циклу событий. Завершаются они в порядке возрастания количества
вычислений: `C`, затем `B`, затем `A`.

### №2
Запустим запросы к нескольким сервисам определения IP параллельно и выберем
первый успешный ответ.

In [3]:
# flake8: noqa
async def get_first_ip():
    services = {
        "api.ipify.org": "https://api.ipify.org?format=json",
        "ip-api.com": "http://ip-api.com/json/",
        "ifconfig.me": "https://ifconfig.me/all.json",
    }

    async def fetch_ip(service, url):
        data = await asyncio.to_thread(fetch_json_url, url)
        ip = data.get("ip") or data.get("query") or data.get("ip_addr")
        return service, ip

    tasks = [
        asyncio.create_task(fetch_ip(name, url))
        for name, url in services.items()
    ]
    try:
        for completed in asyncio.as_completed(tasks):
            try:
                service, ip = await completed
                if ip:
                    return service, ip
            except Exception:
                continue
        raise RuntimeError("Ни один сервис не вернул IP-адрес")
    finally:
        for task in tasks:
            task.cancel()


async def show_first_ip():
    try:
        service, ip = await get_first_ip()
        print(f"Первым ответил {service}: {ip}")
    except Exception as exc:
        print(exc)


run_async(show_first_ip())


<Task pending name='Task-38' coro=<show_first_ip() running at /var/folders/kp/d5m_y7fs5f9dzqchhd9s80hw0000gn/T/ipykernel_54422/3710200892.py:32>>

Первым ответил api.ipify.org: 93.152.217.68


### №3
Реализуем асинхронную функцию `interviews()`, которая принимает произвольное
число претендентов. Все времена ожидания делятся на `100`.

In [ ]:
# flake8: noqa
async def interview_candidate(
    name, prepare_1, defense_1, prepare_2, defense_2
):
    print(f"{name} started the 1 task.")
    await asyncio.sleep(prepare_1 / 100)
    print(f"{name} moved on to the defense of the 1 task.")
    await asyncio.sleep(defense_1 / 100)
    print(f"{name} completed the 1 task.")
    print(f"{name} is resting.")
    await asyncio.sleep(5 / 100)
    print(f"{name} started the 2 task.")
    await asyncio.sleep(prepare_2 / 100)
    print(f"{name} moved on to the defense of the 2 task.")
    await asyncio.sleep(defense_2 / 100)
    print(f"{name} completed the 2 task.")


async def interviews(*candidates):
    await asyncio.gather(
        *(interview_candidate(*candidate) for candidate in candidates)
    )


run_async(
    interviews(
        ("Ivan", 300, 120, 240, 100),
        ("Anna", 220, 160, 180, 140),
        ("Petr", 260, 110, 200, 130),
    )
)


<Task pending name='Task-45' coro=<interviews() running at /var/folders/kp/d5m_y7fs5f9dzqchhd9s80hw0000gn/T/ipykernel_54422/2798090313.py:19>>

Ivan started the 1 task.
Anna started the 1 task.
Petr started the 1 task.
Anna moved on to the defense of the 1 task.
Petr moved on to the defense of the 1 task.
Ivan moved on to the defense of the 1 task.
Petr completed the 1 task.
Petr is resting.
Petr started the 2 task.
Anna completed the 1 task.
Anna is resting.
Anna started the 2 task.
Ivan completed the 1 task.
Ivan is resting.
Ivan started the 2 task.
Anna moved on to the defense of the 2 task.
Petr moved on to the defense of the 2 task.
Ivan moved on to the defense of the 2 task.
Anna completed the 2 task.
Petr completed the 2 task.
Ivan completed the 2 task.


### №4
Основные этапы выращивания рассады идут строго последовательно, а подкормка
и обработка от вредителей запускаются как обязательные параллельные задачи.
Все времена ожидания делятся на `1000`.

In [5]:
# flake8: noqa
async def sow_one(plant, soaking, germination, rooting):
    async def fertilize():
        print(f"7 Application of fertilizers for {plant}")
        await asyncio.sleep(3 / 1000)
        print(f"7 Fertilizers for the {plant} have been introduced")

    async def treat():
        print(f"8 Treatment of {plant} from pests")
        await asyncio.sleep(5 / 1000)
        print(f"8 The {plant} is treated from pests")

    print(f"0 Beginning of sowing the {plant} plant")
    extra_jobs = [
        asyncio.create_task(fertilize()),
        asyncio.create_task(treat()),
    ]
    print(f"1 Soaking of the {plant} started")
    await asyncio.sleep(soaking / 1000)
    print(f"2 Soaking of the {plant} is finished")
    print(f"3 Shelter of the {plant} is supplied")
    await asyncio.sleep(germination / 1000)
    print(f"4 Shelter of the {plant} is removed")
    print(f"5 The {plant} has been transplanted")
    await asyncio.sleep(rooting / 1000)
    print(f"6 The {plant} has taken root")
    await asyncio.gather(*extra_jobs)
    print(f"9 The seedlings of the {plant} are ready")


async def sowing(*plants):
    await asyncio.gather(*(sow_one(*plant) for plant in plants))


run_async(
    sowing(
        ("tomato", 400, 900, 300),
        ("pepper", 500, 700, 350),
        ("cucumber", 250, 450, 200),
    )
)


<Task pending name='Task-52' coro=<sowing() running at /var/folders/kp/d5m_y7fs5f9dzqchhd9s80hw0000gn/T/ipykernel_54422/3387415013.py:31>>

0 Beginning of sowing the tomato plant
1 Soaking of the tomato started
0 Beginning of sowing the pepper plant
1 Soaking of the pepper started
0 Beginning of sowing the cucumber plant
1 Soaking of the cucumber started
7 Application of fertilizers for tomato
8 Treatment of tomato from pests
7 Application of fertilizers for pepper
8 Treatment of pepper from pests
7 Application of fertilizers for cucumber
8 Treatment of cucumber from pests
7 Fertilizers for the tomato have been introduced
7 Fertilizers for the pepper have been introduced
7 Fertilizers for the cucumber have been introduced
8 The tomato is treated from pests
8 The pepper is treated from pests
8 The cucumber is treated from pests
2 Soaking of the cucumber is finished
3 Shelter of the cucumber is supplied
2 Soaking of the tomato is finished
3 Shelter of the tomato is supplied
2 Soaking of the pepper is finished
3 Shelter of the pepper is supplied
4 Shelter of the cucumber is removed
5 The cucumber has been transplanted
6 The c

## Telegram

Задания №5-№12 вынесены в папку `telegram_bots`, потому что боты
нужно запускать отдельным процессом, а не из ячейки ноутбука.

Файлы:

- `telegram_bots/main.py` - запуск нужного бота.
- `telegram_bots/factories.py` - обработчики и клавиатуры.
- `telegram_bots/services.py` - API, переводчик, парсер товаров.
- `telegram_bots/config.py` - чтение `.env`.
- `telegram_bots/quiz_questions.json` - вопросы для задания №9.

Установка зависимостей:

```bash
pip install -r requirements.txt
```

Запуск:

```bash
python telegram_bots/main.py echo
python telegram_bots/main.py time
python telegram_bots/main.py board
python telegram_bots/main.py museum
python telegram_bots/main.py quiz
python telegram_bots/main.py geocoder
python telegram_bots/main.py translator
python telegram_bots/main.py price
```


In [ ]:
# flake8: noqa
commands = [
    "python telegram_bots/main.py echo",
    "python telegram_bots/main.py time",
    "python telegram_bots/main.py board",
    "python telegram_bots/main.py museum",
    "python telegram_bots/main.py quiz",
    "python telegram_bots/main.py geocoder",
    "python telegram_bots/main.py translator",
    "python telegram_bots/main.py price",
]

print(*commands, sep="\n")


### Диаграмма для задания №8

```mermaid
stateDiagram-v2
    state "Вход" as entrance
    state "Зал 1" as room1
    state "Зал 2" as room2
    state "Зал 3" as room3
    state "Зал 4" as room4
    state "Выход" as museum_exit
    [*] --> entrance
    entrance --> room1
    room1 --> room2
    room1 --> room3
    room2 --> room4
    room3 --> room4
    room4 --> museum_exit
    museum_exit --> [*]
```
